In [11]:
import numpy as np 
import pandas as pd 

import os

In [12]:
import json
import os
import pandas as pd
import numpy as np

DATA_DIR = '../input/competitions/cassava-leaf-disease-classification/'

with open(os.path.join(DATA_DIR, 'label_num_to_disease_map.json')) as f:
    disease_map = json.load(f)

df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))

# 숫자로 된 label을 실제 질병 이름으로 바꾼 열을 하나 더 추가
df['disease_name'] = df['label'].astype(str).map(disease_map)

### 데이터 증강(Augmentation) 설정
> **Note:** 카사바 잎 데이터의 특성(배경 노이즈, 다양한 촬영 거리)을 고려하여  
> 단순 리사이즈가 아닌 **무작위 크롭(RandomResizedCrop)** 을 핵심 전략으로 사용합니다.

In [13]:
import albumentations as A
from albumentations.pytorch import ToTensorV2


train_transform = A.Compose([
    A.RandomResizedCrop(size=(256, 256), p=1.0), # 확률 100%로 사이즈 256*256으로 무작위 확대 및 자르기
    A.HorizontalFlip(p=0.5),      # 좌우 반전
    A.VerticalFlip(p=0.5),        # 상하 반전
    A.ShiftScaleRotate(rotate_limit=20, p=0.5), # -20~+20도 사이 랜덤 회전으로 각도 다양성 확보

    # 색감 변화
    A.HueSaturationValue(hue_shift_limit=0.2, sat_shift_limit=0.2, val_shift_limit=0.2, p=0.5), # 색상, 채도, 명도를 최대 20% 범위 내에서 변경

    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # 정규화
    ToTensorV2() # Tensor 변환
])


test_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [14]:
from sklearn.model_selection import train_test_split


train_df, test_df = train_test_split(df, test_size=0.2, random_state=42,
                                     stratify=df['label']) #클래스 불균형이 발생했으므로 label의 개수를 동일하게 split

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"학습용 데이터 개수: {len(train_df)}장")
print(f"테스트용 데이터 개수: {len(test_df)}장")

학습용 데이터 개수: 17117장
테스트용 데이터 개수: 4280장


In [22]:
import torch
from torch.utils.data import Dataset
import cv2

class CassavaDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)


    def __getitem__(self, index):
      image_name = self.df.loc[index, 'image_id']
      label = self.df.loc[index, 'label']

      image_path = os.path.join(self.image_dir, image_name)
      image = cv2.imread(image_path)
      image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

      if self.transform is not None:
          augmented = self.transform(image=image)
          image = augmented['image']

      return image, torch.tensor(label, dtype=torch.long)

> **[구조 변경]** Albumentations 라이브러리는 변환 결과를 딕셔너리 형태로 반환하므로,  
> `self.transform(image=image)['image']` 형식을 사용하여 텐서 데이터를 추출하도록 수정

In [16]:
from torch.utils.data import DataLoader

train_dataset = CassavaDataset(df=train_df, image_dir=DATA_DIR+'train_images', transform=train_transform)
test_dataset = CassavaDataset(df=test_df, image_dir=DATA_DIR+'train_images', transform=test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [17]:
from torchvision import models
import torch.nn as nn

# ResNet 모델 load
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 183MB/s] 


In [18]:
# 출력층 개수 변경

n_features = model.fc.in_features
model.fc = nn.Linear(n_features, 5)

In [19]:
# 모델 GPU로 전송

if torch.cuda.is_available():
    print("yes")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

yes


In [20]:
#손실함수와 옵티마이저 설정

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-6)

In [25]:
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        outputs = model(images)
        
        loss = criterion(outputs, labels)

        #오차역전파
        loss.backward()

        #가중치 수정
        optimizer.step()

        running_loss += loss.item()
        
    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] 완료! - 평균 오답 점수(Loss): {epoch_loss:.4f}")
print("학습 완료")

Epoch [1/20] 완료! - 평균 오답 점수(Loss): 0.4021
Epoch [2/20] 완료! - 평균 오답 점수(Loss): 0.3963
Epoch [3/20] 완료! - 평균 오답 점수(Loss): 0.3923
Epoch [4/20] 완료! - 평균 오답 점수(Loss): 0.3844
Epoch [5/20] 완료! - 평균 오답 점수(Loss): 0.3786
Epoch [6/20] 완료! - 평균 오답 점수(Loss): 0.3850
Epoch [7/20] 완료! - 평균 오답 점수(Loss): 0.3677
Epoch [8/20] 완료! - 평균 오답 점수(Loss): 0.3641
Epoch [9/20] 완료! - 평균 오답 점수(Loss): 0.3600
Epoch [10/20] 완료! - 평균 오답 점수(Loss): 0.3584
Epoch [11/20] 완료! - 평균 오답 점수(Loss): 0.3521
Epoch [12/20] 완료! - 평균 오답 점수(Loss): 0.3542
Epoch [13/20] 완료! - 평균 오답 점수(Loss): 0.3445
Epoch [14/20] 완료! - 평균 오답 점수(Loss): 0.3446
Epoch [15/20] 완료! - 평균 오답 점수(Loss): 0.3332
Epoch [16/20] 완료! - 평균 오답 점수(Loss): 0.3327
Epoch [17/20] 완료! - 평균 오답 점수(Loss): 0.3282
Epoch [18/20] 완료! - 평균 오답 점수(Loss): 0.3285
Epoch [19/20] 완료! - 평균 오답 점수(Loss): 0.3162
Epoch [20/20] 완료! - 평균 오답 점수(Loss): 0.3154
학습 완료


In [26]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

final_accuracy = accuracy_score(all_labels, all_preds)
print(f"/n Test 정확도: {final_accuracy * 100:.2f}%")

print("\n Classification Report")

disease_names = ['CBB (0)', 'CBSD (1)', 'CGM (2)', 'CMD (3)', 'Healthy (4)']
print(classification_report(all_labels, all_preds, target_names=disease_names))

/n Test 정확도: 84.65%

 Classification Report
              precision    recall  f1-score   support

     CBB (0)       0.61      0.49      0.55       217
    CBSD (1)       0.76      0.73      0.74       438
     CGM (2)       0.75      0.70      0.72       477
     CMD (3)       0.94      0.95      0.94      2632
 Healthy (4)       0.63      0.73      0.68       516

    accuracy                           0.85      4280
   macro avg       0.74      0.72      0.73      4280
weighted avg       0.85      0.85      0.85      4280

